# 01 Data And Subsets
Rohdaten laden, `mention_id`/`block_key` prüfen, deduplizieren, Stage-Subsets erstellen und QC ausgeben.

In [1]:
from pathlib import Path
import pandas as pd
import sys

RUN_STAGE = "smoke"
RUN_ID = "replace_with_run_id_from_00"
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.common.config import load_yaml, resolve_paths_config
from src.data.prepare_lspo import prepare_lspo_mentions
from src.data.prepare_ads import prepare_ads_mentions
from src.common.subset_builder import build_stage_subset, write_subset_manifest
from src.common.io_schema import save_parquet
from src.common.run_report import summarize_block_distribution

paths_cfg = resolve_paths_config(load_yaml(PROJECT_ROOT / "configs/paths.colab.yaml"), project_root=PROJECT_ROOT)
run_cfg = load_yaml(PROJECT_ROOT / f"configs/runs/{RUN_STAGE}.yaml")

run_dirs = {
    "metrics": PROJECT_ROOT / "artifacts/metrics" / RUN_ID,
    "subset_cache": PROJECT_ROOT / "data/subsets/cache" / RUN_ID,
}
for p in run_dirs.values():
    p.mkdir(parents=True, exist_ok=True)


In [3]:
# Input file checks
from pathlib import Path

raw_checks = {
    "raw_lspo_parquet": Path(paths_cfg["data"]["raw_lspo_parquet"]).exists(),
    "raw_lspo_h5": Path(paths_cfg["data"]["raw_lspo_h5"]).exists(),
    "raw_ads_publications": Path(paths_cfg["data"]["raw_ads_publications"]).exists(),
    "raw_ads_references": Path(paths_cfg["data"]["raw_ads_references"]).exists(),
}
raw_checks


{'raw_lspo_parquet': True,
 'raw_lspo_h5': True,
 'raw_ads_publications': True,
 'raw_ads_references': True}

In [4]:
# LSPO + ADS mentions erzeugen
lspo_mentions = prepare_lspo_mentions(
    parquet_path=paths_cfg["data"]["raw_lspo_parquet"],
    h5_path=paths_cfg["data"].get("raw_lspo_h5"),
    output_path=PROJECT_ROOT / "data/interim/lspo_mentions.parquet",
)
ads_mentions = prepare_ads_mentions(
    publications_path=paths_cfg["data"]["raw_ads_publications"],
    references_path=paths_cfg["data"]["raw_ads_references"],
    output_path=PROJECT_ROOT / "data/interim/ads_mentions.parquet",
)

print("LSPO mentions:", len(lspo_mentions))
print("ADS mentions:", len(ads_mentions))

LSPO mentions: 554962
ADS mentions: 1550512


In [5]:
# Stage-Subsets
stage = run_cfg["stage"]
seed = int(run_cfg.get("seed", 11))
target = run_cfg.get("subset_target_mentions")

lspo_subset = build_stage_subset(lspo_mentions, stage=stage, seed=seed, target_mentions=target)
ads_subset = build_stage_subset(ads_mentions, stage=stage, seed=seed, target_mentions=target)

lspo_subset_path = run_dirs["subset_cache"] / f"lspo_mentions_{stage}.parquet"
ads_subset_path = run_dirs["subset_cache"] / f"ads_mentions_{stage}.parquet"
lspo_manifest_path = PROJECT_ROOT / "data/subsets/manifests" / f"{RUN_ID}_lspo_{stage}_manifest.parquet"
ads_manifest_path = PROJECT_ROOT / "data/subsets/manifests" / f"{RUN_ID}_ads_{stage}_manifest.parquet"

save_parquet(lspo_subset, lspo_subset_path, index=False)
save_parquet(ads_subset, ads_subset_path, index=False)
write_subset_manifest(lspo_subset, lspo_manifest_path)
write_subset_manifest(ads_subset, ads_manifest_path)

print("LSPO subset:", len(lspo_subset), "->", lspo_subset_path)
print("ADS subset:", len(ads_subset), "->", ads_subset_path)


LSPO subset: 1000 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\replace_with_run_id_from_00\lspo_mentions_smoke.parquet
ADS subset: 1000 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\replace_with_run_id_from_00\ads_mentions_smoke.parquet


In [6]:
# QC: Counts + Missingness + Top-ambige Blöcke

def qc_table(df, name):
    return {
        "name": name,
        "records": len(df),
        "mentions": df["mention_id"].nunique(),
        "bibcodes": df["bibcode"].nunique(),
        "blocks": df["block_key"].nunique(),
        "missing_title": int(df["title"].isna().sum() + (df["title"].astype(str).str.strip()=="").sum()),
        "missing_abstract": int(df["abstract"].isna().sum() + (df["abstract"].astype(str).str.strip()=="").sum()),
        "missing_year": int(df["year"].isna().sum()),
        "missing_author": int(df["author_raw"].isna().sum() + (df["author_raw"].astype(str).str.strip()=="").sum()),
    }

qc = pd.DataFrame([
    qc_table(lspo_subset, "lspo_subset"),
    qc_table(ads_subset, "ads_subset"),
])
qc


,name,records,mentions,bibcodes,blocks,missing_title,missing_abstract,missing_year,missing_author
0,lspo_subset,1000,1000,1000,500,0,0,1000,0
1,ads_subset,1000,1000,985,500,0,128,0,0


In [7]:
print("Top-20 ambige LSPO-Blöcke")
display(summarize_block_distribution(lspo_subset, top_k=20))
print("Top-20 ambige ADS-Blöcke")
display(summarize_block_distribution(ads_subset, top_k=20))


Top-20 ambige LSPO-Blöcke


TypeError: Series.reset_index() got an unexpected keyword argument 'names'